# 🍵 RAG Tea Knowledge System - Part 3

## Evaluation & Metrics

This notebook evaluates the RAG retrieval system using standard IR metrics.

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import json
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries loaded")

---
## 1. Evaluation Dataset

Creating test queries with relevant document IDs (ground truth).

In [ ]:
# =============================================================================
# EVALUATION QUERIES WITH GROUND TRUTH
# =============================================================================

EVAL_DATASET = [
    {
        "query_id": "Q01",
        "query": "What is TRI 2025 tea cultivar and its characteristics?",
        "relevant_docs": ["CUL_001"],
        "category": "cultivar"
    },
    {
        "query_id": "Q02",
        "query": "Which tea cultivars are resistant to blister blight disease?",
        "relevant_docs": ["CUL_001", "CUL_005", "CUL_006", "CUL_007", "DIS_001"],
        "category": "cultivar"
    },
    {
        "query_id": "Q03",
        "query": "What are the health benefits of EGCG in tea?",
        "relevant_docs": ["HLT_001", "HLT_002"],
        "category": "health"
    },
    {
        "query_id": "Q04",
        "query": "Describe Nuwara Eliya tea region and its characteristics",
        "relevant_docs": ["REG_001"],
        "category": "region"
    },
    {
        "query_id": "Q05",
        "query": "What is the difference between orthodox and CTC processing?",
        "relevant_docs": ["PRO_001", "PRO_002"],
        "category": "processing"
    },
    {
        "query_id": "Q06",
        "query": "What is Orange Pekoe tea grade?",
        "relevant_docs": ["GRD_001", "GRD_002"],
        "category": "grade"
    },
    {
        "query_id": "Q07",
        "query": "How does elevation affect tea quality?",
        "relevant_docs": ["REG_008", "QUA_002"],
        "category": "quality"
    },
    {
        "query_id": "Q08",
        "query": "What is fine plucking standard for tea?",
        "relevant_docs": ["PLK_001", "PLK_002"],
        "category": "plucking"
    },
    {
        "query_id": "Q09",
        "query": "What causes blister blight disease in tea?",
        "relevant_docs": ["DIS_001"],
        "category": "disease"
    },
    {
        "query_id": "Q10",
        "query": "What is L-theanine and its effects?",
        "relevant_docs": ["HLT_003"],
        "category": "health"
    },
    {
        "query_id": "Q11",
        "query": "How is Silver Tips tea produced?",
        "relevant_docs": ["GRD_010", "PLK_002"],
        "category": "grade"
    },
    {
        "query_id": "Q12",
        "query": "What is the Uva tea region known for?",
        "relevant_docs": ["REG_003"],
        "category": "region"
    },
    {
        "query_id": "Q13",
        "query": "What are nematode tolerant tea cultivars?",
        "relevant_docs": ["CUL_008", "CUL_009", "CUL_012", "DIS_002"],
        "category": "cultivar"
    },
    {
        "query_id": "Q14",
        "query": "How does withering affect tea quality?",
        "relevant_docs": ["PRO_003", "PRO_001"],
        "category": "processing"
    },
    {
        "query_id": "Q15",
        "query": "What is the AI grading system for tea leaves?",
        "relevant_docs": ["AI_001", "AI_002", "AI_003"],
        "category": "ai_grading"
    },
    {
        "query_id": "Q16",
        "query": "What tea grades are best for tea bags?",
        "relevant_docs": ["GRD_008", "GRD_009", "GRD_013"],
        "category": "grade"
    },
    {
        "query_id": "Q17",
        "query": "How much tea should I drink daily for health?",
        "relevant_docs": ["HLT_007", "HLT_004"],
        "category": "health"
    },
    {
        "query_id": "Q18",
        "query": "What is the history of Ceylon tea?",
        "relevant_docs": ["HIS_001", "HIS_002"],
        "category": "history"
    },
    {
        "query_id": "Q19",
        "query": "How does climate change affect tea production?",
        "relevant_docs": ["SUS_002"],
        "category": "sustainability"
    },
    {
        "query_id": "Q20",
        "query": "What is the Ceylon tea Lion Logo certification?",
        "relevant_docs": ["TRA_002", "REG_008"],
        "category": "trade"
    }
]

print(f"✅ Created evaluation dataset with {len(EVAL_DATASET)} queries")

# Save evaluation dataset
with open('./data/processed/eval_dataset.json', 'w') as f:
    json.dump(EVAL_DATASET, f, indent=2)

print(f"💾 Saved to ./data/processed/eval_dataset.json")

---
## 2. Evaluation Metrics

In [ ]:
# =============================================================================
# EVALUATION METRICS
# =============================================================================

class RetrievalEvaluator:
    """
    Evaluate retrieval system using standard IR metrics.
    
    Metrics:
    - Precision@K
    - Recall@K
    - MRR (Mean Reciprocal Rank)
    - nDCG@K (Normalized Discounted Cumulative Gain)
    - Hit Rate@K
    """
    
    def precision_at_k(self, retrieved: List[str], relevant: List[str], k: int) -> float:
        """Precision@K: fraction of retrieved docs that are relevant"""
        retrieved_k = retrieved[:k]
        relevant_retrieved = len(set(retrieved_k) & set(relevant))
        return relevant_retrieved / k if k > 0 else 0.0
    
    def recall_at_k(self, retrieved: List[str], relevant: List[str], k: int) -> float:
        """Recall@K: fraction of relevant docs that are retrieved"""
        retrieved_k = retrieved[:k]
        relevant_retrieved = len(set(retrieved_k) & set(relevant))
        return relevant_retrieved / len(relevant) if len(relevant) > 0 else 0.0
    
    def mrr(self, retrieved: List[str], relevant: List[str]) -> float:
        """Mean Reciprocal Rank: 1/rank of first relevant doc"""
        for i, doc in enumerate(retrieved):
            if doc in relevant:
                return 1.0 / (i + 1)
        return 0.0
    
    def ndcg_at_k(self, retrieved: List[str], relevant: List[str], k: int) -> float:
        """Normalized Discounted Cumulative Gain@K"""
        dcg = 0.0
        for i, doc in enumerate(retrieved[:k]):
            if doc in relevant:
                dcg += 1.0 / np.log2(i + 2)  # +2 because log2(1)=0
        
        # Ideal DCG (all relevant docs at top)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
        
        return dcg / idcg if idcg > 0 else 0.0
    
    def hit_rate_at_k(self, retrieved: List[str], relevant: List[str], k: int) -> float:
        """Hit Rate@K: 1 if any relevant doc in top-k, else 0"""
        retrieved_k = set(retrieved[:k])
        return 1.0 if len(retrieved_k & set(relevant)) > 0 else 0.0
    
    def evaluate_single(self, retrieved: List[str], relevant: List[str], k_values: List[int] = [1, 3, 5, 10]) -> Dict:
        """Evaluate single query across all metrics"""
        results = {'mrr': self.mrr(retrieved, relevant)}
        
        for k in k_values:
            results[f'precision@{k}'] = self.precision_at_k(retrieved, relevant, k)
            results[f'recall@{k}'] = self.recall_at_k(retrieved, relevant, k)
            results[f'ndcg@{k}'] = self.ndcg_at_k(retrieved, relevant, k)
            results[f'hit_rate@{k}'] = self.hit_rate_at_k(retrieved, relevant, k)
        
        return results
    
    def evaluate_batch(self, results_list: List[Tuple[List[str], List[str]]], 
                       k_values: List[int] = [1, 3, 5, 10]) -> Dict:
        """Evaluate batch of queries, return averaged metrics"""
        all_metrics = []
        
        for retrieved, relevant in results_list:
            metrics = self.evaluate_single(retrieved, relevant, k_values)
            all_metrics.append(metrics)
        
        # Average all metrics
        avg_metrics = {}
        for key in all_metrics[0].keys():
            avg_metrics[key] = np.mean([m[key] for m in all_metrics])
        
        return avg_metrics

evaluator = RetrievalEvaluator()
print("✅ Evaluator initialized")

In [ ]:
# =============================================================================
# MOCK EVALUATION (Replace with actual retrieval results)
# =============================================================================

# Simulated retrieval results for demonstration
# In practice, you would run the actual retriever and get doc_ids

mock_results = {
    'dense': [
        (["CUL_001", "CUL_002", "CUL_003", "CUL_004", "CUL_005"], ["CUL_001"]),  # Q01
        (["CUL_006", "CUL_001", "DIS_001", "CUL_007", "CUL_005"], ["CUL_001", "CUL_005", "CUL_006", "CUL_007", "DIS_001"]),  # Q02
        (["HLT_001", "HLT_002", "HLT_004", "HLT_003", "HLT_006"], ["HLT_001", "HLT_002"]),  # Q03
        (["REG_001", "REG_002", "REG_008", "REG_003", "REG_004"], ["REG_001"]),  # Q04
        (["PRO_001", "PRO_002", "PRO_004", "PRO_003", "PRO_005"], ["PRO_001", "PRO_002"]),  # Q05
    ],
    'bm25': [
        (["CUL_001", "CUL_003", "CUL_002", "CUL_005", "CUL_004"], ["CUL_001"]),  # Q01
        (["DIS_001", "CUL_001", "CUL_006", "CUL_005", "CUL_007"], ["CUL_001", "CUL_005", "CUL_006", "CUL_007", "DIS_001"]),  # Q02
        (["HLT_001", "HLT_006", "HLT_002", "HLT_004", "HLT_003"], ["HLT_001", "HLT_002"]),  # Q03
        (["REG_001", "REG_008", "REG_002", "REG_004", "REG_003"], ["REG_001"]),  # Q04
        (["PRO_002", "PRO_001", "PRO_004", "PRO_003", "PRO_006"], ["PRO_001", "PRO_002"]),  # Q05
    ],
    'hybrid': [
        (["CUL_001", "CUL_002", "CUL_003", "CUL_005", "CUL_004"], ["CUL_001"]),  # Q01
        (["CUL_001", "DIS_001", "CUL_006", "CUL_007", "CUL_005"], ["CUL_001", "CUL_005", "CUL_006", "CUL_007", "DIS_001"]),  # Q02
        (["HLT_001", "HLT_002", "HLT_004", "HLT_006", "HLT_003"], ["HLT_001", "HLT_002"]),  # Q03
        (["REG_001", "REG_002", "REG_008", "REG_003", "REG_004"], ["REG_001"]),  # Q04
        (["PRO_001", "PRO_002", "PRO_004", "PRO_003", "PRO_005"], ["PRO_001", "PRO_002"]),  # Q05
    ]
}

# Evaluate each method
print("📊 EVALUATION RESULTS")
print("=" * 70)

eval_results = {}
for method, results in mock_results.items():
    metrics = evaluator.evaluate_batch(results, k_values=[1, 3, 5])
    eval_results[method] = metrics
    
    print(f"\n📌 {method.upper()}")
    print(f"   MRR: {metrics['mrr']:.4f}")
    print(f"   Precision@1: {metrics['precision@1']:.4f}")
    print(f"   Recall@5: {metrics['recall@5']:.4f}")
    print(f"   nDCG@5: {metrics['ndcg@5']:.4f}")
    print(f"   Hit Rate@5: {metrics['hit_rate@5']:.4f}")

In [ ]:
# =============================================================================
# VISUALIZE COMPARISON
# =============================================================================

# Create comparison dataframe
comparison_data = []
for method, metrics in eval_results.items():
    for metric_name, value in metrics.items():
        comparison_data.append({
            'Method': method.upper(),
            'Metric': metric_name,
            'Value': value
        })

comparison_df = pd.DataFrame(comparison_data)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Key metrics comparison
key_metrics = ['mrr', 'precision@1', 'recall@5', 'ndcg@5', 'hit_rate@5']
key_df = comparison_df[comparison_df['Metric'].isin(key_metrics)]

pivot_df = key_df.pivot(index='Metric', columns='Method', values='Value')
pivot_df.plot(kind='bar', ax=axes[0], rot=45)
axes[0].set_ylabel('Score')
axes[0].set_title('Retrieval Methods Comparison')
axes[0].legend(title='Method')
axes[0].set_ylim(0, 1.1)

# Precision-Recall curve (simulated)
k_values = [1, 3, 5]
for method in ['DENSE', 'BM25', 'HYBRID']:
    precisions = [eval_results[method.lower()][f'precision@{k}'] for k in k_values]
    recalls = [eval_results[method.lower()][f'recall@{k}'] for k in k_values]
    axes[1].plot(recalls, precisions, 'o-', label=method, markersize=8)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Trade-off')
axes[1].legend()
axes[1].set_xlim(0, 1.1)
axes[1].set_ylim(0, 1.1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('./results/evaluation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Saved: ./results/evaluation_comparison.png")

In [ ]:
# =============================================================================
# SAVE EVALUATION RESULTS
# =============================================================================

# Save to JSON
with open('./results/evaluation_metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

# Save to CSV
comparison_df.to_csv('./results/evaluation_comparison.csv', index=False)

print("💾 Saved evaluation results:")
print("   - ./results/evaluation_metrics.json")
print("   - ./results/evaluation_comparison.csv")
print("   - ./results/evaluation_comparison.png")

---
## Summary

### Evaluation Metrics Used:
- **MRR** (Mean Reciprocal Rank): Measures how early the first relevant doc appears
- **Precision@K**: Fraction of top-K results that are relevant
- **Recall@K**: Fraction of relevant docs found in top-K
- **nDCG@K**: Considers both relevance and ranking position
- **Hit Rate@K**: Binary success metric

### Key Findings:
- Hybrid search typically outperforms single methods
- Dense retrieval better for semantic queries
- BM25 better for keyword-specific queries
- Combination provides robust performance